# Poisson ML Regression

In this notebook, we will implement a ML model to calculate a suitable lambda value for a Poisson distribution modelling number of goals for a team. We will build it using inputs Elo diff and Home advantage.

## Data prep 

We have our Elo calculation in place. We seek to find the relationship via regression between Elo diff, home ad, and expected number of goals. Currently, our Elo calculation produces a final Elo dictionary for each team, so we need to alter the logic to be able to retrieve the Elo before each match.

In [3]:
import sqlite3
import pandas as pd
import soccerdata as sd

connection = sqlite3.connect("../data/processed/football.db")

matches = pd.read_sql_query(
    "SELECT * FROM matches",
    connection
)

matches["date"] = pd.to_datetime(matches["date"])
matches = matches.sort_values("date").reset_index(drop=True)

In [4]:
clubelo = sd.ClubElo()

elo_2015 = clubelo.read_by_date("2015-06-30")

countries = ["ENG", "SCO", "GER", "ESP", "ITA", "FRA", "BEL", "NED", "POR", "TUR", "GRE"]

top_leagues = elo_2015[
    (elo_2015["country"].isin(countries)) &
    (elo_2015["level"] == 1)
]

league_elos = top_leagues.groupby("country")["elo"].mean()

league_to_country = {
    "E0": "ENG",
    "SC0": "SCO",
    "D1": "GER",
    "SP1": "ESP",
    "I1": "ITA",
    "F1": "FRA",
    "B1": "BEL",
    "N1": "NED",
    "P1": "POR",
    "T1": "TUR",
    "G1": "GRE"
}

team_leagues = pd.read_sql_query(
    "SELECT DISTINCT team_id, league FROM team_aliases WHERE source = 'football_data'",
    connection
)

[08/13/26 16:00:58] INFO     Saving cached data to C:\Users\rohan\soccerdata\data\ClubElo            ]8;id=2644347;file://c:\Users\rohan\Documents\UCL-MU-Predictor\.venv\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=2644348;file://c:\Users\rohan\Documents\UCL-MU-Predictor\.venv\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

[2026-08-13 16:00:58] INFO     TLSLibrary:_load_library:397 - Successfully loaded TLS library: C:\Users\rohan\Documents\UCL-MU-Predictor\.venv\Lib\site-packages\tls_requests\bin\tls-client-xgo-1.13.1-windows-amd64.dll


                    INFO     Successfully loaded TLS library:                                      ]8;id=2644355;file://c:\Users\rohan\Documents\UCL-MU-Predictor\.venv\Lib\site-packages\tls_requests\models\libraries.py\libraries.py]8;;\:]8;id=2644356;file://c:\Users\rohan\Documents\UCL-MU-Predictor\.venv\Lib\site-packages\tls_requests\models\libraries.py#397\397]8;;\
                             C:\Users\rohan\Documents\UCL-MU-Predictor\.venv\Lib\site-packages\tls                 
                             _requests\bin\tls-client-xgo-1.13.1-windows-amd64.dll                                 

In [ ]:
# building auxilliary table (key match_id) to matches, with per match elo data (elo before, elo after etc.)
# join to full table, to give full match data with elo 

elo = {}

for _, team in team_leagues.iterrows():

    league = team["league"]
    country = league_to_country[league]

    elo[team["team_id"]] = league_elos.loc[country]


home_elo_before = []
away_elo_before = []

home_elo_after = []
away_elo_after = []

expected_home_scores = []
expected_away_scores = []

K = 20

for _, match in matches.iterrows():

    if match["home_team_id"] not in elo:
        elo[match["home_team_id"]] = 1500

    if match["away_team_id"] not in elo:
        elo[match["away_team_id"]] = 1500

    current_home_rating = elo[match["home_team_id"]]
    current_away_rating = elo[match["away_team_id"]]

    home_elo_before.append(current_home_rating)
    away_elo_before.append(current_away_rating)

    Q_A = 10 ** (current_home_rating / 400)
    Q_B = 10 ** (current_away_rating / 400)

    expected_home_score = Q_A / (Q_A + Q_B)
    expected_away_score = Q_B / (Q_A + Q_B)

    expected_home_scores.append(expected_home_score)
    expected_away_scores.append(expected_away_score)

    if match["home_goals"] > match["away_goals"]:
        S_A = 1
        S_B = 0

    elif match["home_goals"] == match["away_goals"]:
        S_A = 0.5
        S_B = 0.5

    else:
        S_A = 0
        S_B = 1

    updated_home_rating = current_home_rating + K * (S_A - expected_home_score)
    updated_away_rating = current_away_rating + K * (S_B - expected_away_score)

    elo[match["home_team_id"]] = updated_home_rating
    elo[match["away_team_id"]] = updated_away_rating

    home_elo_after.append(updated_home_rating)
    away_elo_after.append(updated_away_rating)


elo_matches = matches.copy()

elo_matches["home_elo_before"] = home_elo_before
elo_matches["away_elo_before"] = away_elo_before

elo_matches["home_elo_after"] = home_elo_after
elo_matches["away_elo_after"] = away_elo_after

elo_matches["expected_home_score"] = expected_home_scores
elo_matches["expected_away_score"] = expected_away_scores

elo_matches["elo_diff"] = (
    elo_matches["home_elo_before"] - elo_matches["away_elo_before"]
)

elo_matches.head()

,match_id,date,season,competition,home_team_id,away_team_id,home_team_name,away_team_name,home_goals,away_goals,source,home_elo_before,away_elo_before,home_elo_after,away_elo_after,expected_home_score,expected_away_score,elo_diff
0,1,2015-06-30,2015/16,CL,14,384,Pyunik,Folgore,2,1,footystats,1500.0,1500.0,1510.0,1490.0,0.5,0.5,0.0
1,2,2015-06-30,2015/16,CL,184,8,Lincoln Red Imps,FC Santa Coloma,0,0,footystats,1500.0,1500.0,1500.0,1500.0,0.5,0.5,0.0
2,3,2015-06-30,2015/16,CL,326,104,Crusaders,Levadia Tallinn,0,0,footystats,1500.0,1500.0,1500.0,1500.0,0.5,0.5,0.0
3,4,2015-07-01,2015/16,CL,106,503,B36 Torshavn,The New Saints,1,2,footystats,1500.0,1500.0,1490.0,1510.0,0.5,0.5,0.0
4,5,2015-07-07,2015/16,CL,8,184,FC Santa Coloma,Lincoln Red Imps,1,2,footystats,1500.0,1500.0,1490.0,1510.0,0.5,0.5,0.0


## Machine Learning - Poisson regression between Elo difference and expected number of goals scored and home advantage

For each match we make some observations:

For the home team:
1. $d = R_H - R_A$ (elo diff)
2. $h = 1$
3. $Y = \text{home goals}$

For the away team:
1. $d = R_A - R_H$ (elo diff)
2. $h = 0$
3. $Y = \text{away goals}$

Where $R_H$ and $R_A$ are the Elo ratings BEFORE the match.

We make the natural assumption that goals follow the Poisson distribution, that is, 

$$
Y_i \sim \text{Poisson}(\lambda_{i}), \\
\mathbb{P}(Y_i = y) = \frac{e^{-\lambda_{i}}\lambda_{i}^{y}}{y!}
$$

We want to use regression to learn what this $\lambda_{i}$ is. As such, we model it as:

$$
log(\lambda_i) = \beta_0 + \beta_1 d_i + \beta_2 h_i, 
$$
or equivalently,
$$
\lambda_i = \exp{(\beta_0 + \beta_1 d_i + \beta_2 h_i)}.
$$

Here,
- $\beta_0$ is a "baseline" scoring level
- $\beta_1$ is the effect of Elo difference
- $\beta_2$ is the effect of home advantage.

The model chooses the beta's by maximising the Poisson likelihood (MLE). The formula is:

$$ 
\ell(\beta) = \sum_{i}{y_i log(\lambda_i) - \lambda_i - log(y_{i}!)}.
$$

Then, once fitted, our values can be calculated as 
$$
\lambda_{H} = \exp{(\beta_0 + \beta_1(R_H - R_A) + \beta_2)} \\
\lambda_{A} = \exp{(\beta_0 + \beta_1(R_A - R_H))}
$$

From this, we can then sample integers scores, and implement a simulation of multiple CL tournaments.

#### Adding observations to data

We build a home version of the table with suitable home observations, and an away version of the table with suitable away observations, then take the Union

In [6]:
elo_matches.head()

,match_id,date,season,competition,home_team_id,away_team_id,home_team_name,away_team_name,home_goals,away_goals,source,home_elo_before,away_elo_before,home_elo_after,away_elo_after,expected_home_score,expected_away_score,elo_diff
0,1,2015-06-30,2015/16,CL,14,384,Pyunik,Folgore,2,1,footystats,1500.0,1500.0,1510.0,1490.0,0.5,0.5,0.0
1,2,2015-06-30,2015/16,CL,184,8,Lincoln Red Imps,FC Santa Coloma,0,0,footystats,1500.0,1500.0,1500.0,1500.0,0.5,0.5,0.0
2,3,2015-06-30,2015/16,CL,326,104,Crusaders,Levadia Tallinn,0,0,footystats,1500.0,1500.0,1500.0,1500.0,0.5,0.5,0.0
3,4,2015-07-01,2015/16,CL,106,503,B36 Torshavn,The New Saints,1,2,footystats,1500.0,1500.0,1490.0,1510.0,0.5,0.5,0.0
4,5,2015-07-07,2015/16,CL,8,184,FC Santa Coloma,Lincoln Red Imps,1,2,footystats,1500.0,1500.0,1490.0,1510.0,0.5,0.5,0.0


In [27]:
home_elo_matches = elo_matches.copy()
home_elo_matches['elo_diff'] = home_elo_matches['home_elo_before'] - home_elo_matches['away_elo_before']
home_elo_matches['H'] = 1
home_elo_matches['goals'] = home_elo_matches['home_goals']

away_elo_matches = elo_matches.copy()
away_elo_matches['elo_diff'] = away_elo_matches['away_elo_before'] - away_elo_matches['home_elo_before']
away_elo_matches['H'] = 0
away_elo_matches['goals'] = away_elo_matches['away_goals']

doubled_elo_matches = pd.concat([home_elo_matches, away_elo_matches])

doubled_elo_matches = doubled_elo_matches.sort_values(
    ['date', 'match_id']
)

middle = len(doubled_elo_matches) // 2

display(doubled_elo_matches[middle - 5:middle + 5])

,match_id,date,season,competition,home_team_id,away_team_id,home_team_name,away_team_name,home_goals,away_goals,source,home_elo_before,away_elo_before,home_elo_after,away_elo_after,expected_home_score,expected_away_score,elo_diff,H,goals
20345,20354,2021-02-05,2020/21,SP1,417,443,Alaves,Real Valladolid,1,0,football_data,1711.480788,1726.130603,1721.902194,1715.709197,0.478930,0.521070,14.649814,0,0
20376,20355,2021-02-06,2020/21,B1,29,38,Cercle Brugge,Mechelen,0,1,football_data,1323.414062,1458.923613,1317.127904,1465.209771,0.314308,0.685692,-135.509551,1,0
20376,20355,2021-02-06,2020/21,B1,29,38,Cercle Brugge,Mechelen,0,1,football_data,1323.414062,1458.923613,1317.127904,1465.209771,0.314308,0.685692,135.509551,0,1
20374,20356,2021-02-06,2020/21,B1,40,45,Oostende,Sint-Truiden,3,1,football_data,1409.704710,1423.584540,1420.103992,1413.185258,0.480036,0.519964,-13.879829,1,3
20374,20356,2021-02-06,2020/21,B1,40,45,Oostende,Sint-Truiden,3,1,football_data,1409.704710,1423.584540,1420.103992,1413.185258,0.480036,0.519964,13.879829,0,1
20377,20357,2021-02-06,2020/21,B1,46,41,Standard Liege,Oud-Heverlee Leuven,1,1,football_data,1501.865265,1422.744855,1499.626563,1424.983557,0.611935,0.388065,79.120410,1,1
20377,20357,2021-02-06,2020/21,B1,46,41,Standard Liege,Oud-Heverlee Leuven,1,1,football_data,1501.865265,1422.744855,1499.626563,1424.983557,0.611935,0.388065,-79.120410,0,1
20378,20358,2021-02-06,2020/21,B1,48,31,Waasland-Beveren,Club Brugge,0,2,football_data,1323.372341,1684.390634,1321.147642,1686.615333,0.111235,0.888765,-361.018294,1,0
20378,20358,2021-02-06,2020/21,B1,48,31,Waasland-Beveren,Club Brugge,0,2,football_data,1323.372341,1684.390634,1321.147642,1686.615333,0.111235,0.888765,361.018294,0,2
20379,20359,2021-02-06,2020/21,D1,154,182,Augsburg,Wolfsburg,0,2,football_data,1650.750555,1798.415150,1644.762006,1804.403700,0.299427,0.700573,-147.664595,1,0


In [28]:
ml_table = doubled_elo_matches.copy()

ml_table = ml_table[
    [
        "match_id",
        "date",
        "season",
        "competition",
        "home_team_name",
        "away_team_name",
        "elo_diff",
        "H",
        "goals"
    ]
]

ml_table = ml_table[
    ml_table["season"].isin([
        "2017/18",
        "2018/19",
        "2019/20",
        "2020/21",
        "2021/22",
        "2022/23",
        "2023/24",
        "2024/25",
        "2025/26"
    ])
]

ml_table = ml_table.sort_values(["date", "match_id", "H"], ascending=[True, True, False]).reset_index(drop=True)

ml_table.head(20)

,match_id,date,season,competition,home_team_name,away_team_name,elo_diff,H,goals
0,7335,2017-06-27,2017/18,CL,Alashkert,FC Santa Coloma,18.929899,1,1
1,7335,2017-06-27,2017/18,CL,Alashkert,FC Santa Coloma,-18.929899,0,0
2,7336,2017-06-27,2017/18,CL,Vikingur Gota,Trepca'89,0.000000,1,2
3,7336,2017-06-27,2017/18,CL,Vikingur Gota,Trepca'89,0.000000,0,1
4,7337,2017-06-27,2017/18,CL,Hibernians,FCI Tallinn,-0.575011,1,2
5,7337,2017-06-27,2017/18,CL,Hibernians,FCI Tallinn,0.575011,0,0
6,7338,2017-06-27,2017/18,CL,The New Saints,Europa FC,15.800917,1,1
7,7338,2017-06-27,2017/18,CL,The New Saints,Europa FC,-15.800917,0,2
8,7339,2017-06-28,2017/18,CL,Linfield,La Fiorita,0.000000,1,1
9,7339,2017-06-28,2017/18,CL,Linfield,La Fiorita,0.000000,0,0
